<a href="https://colab.research.google.com/github/Abdoulaye279/Financial-Engineering/blob/main/lehman-ai-agents/part2_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lehman 10-Q (July 2008): RAG version

Same question as `part1_full_context.ipynb`, but the 10-Q is cut into chunks stored in Qdrant and only the 10 closest chunks go to the model.

Gemini key: Colab secret `GEMINI_API_KEY`, or env variable `GOOGLE_API_KEY` locally. Filing: [PDF](https://www.rns-pdf.londonstockexchange.com/rns/8436Z_1-2008-7-24.pdf), [SEC text](https://www.sec.gov/Archives/edgar/data/806085/000110465908045115/a08-18147_110q.htm)

In [1]:
!pip install -q edgartools==5.21.1 langchain==1.2.10 langchain-google-genai==4.2.1 langchain-qdrant==1.1.0 langchain-text-splitters==1.1.1 qdrant-client==1.17.0 pydantic==2.12.3 rich==13.9.4

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 1.75.0 which is incompatible.


In [2]:
import os
import time
from typing import List, Optional
from datetime import datetime
from pydantic import BaseModel, Field
from edgar import Company, set_identity
from google import genai
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import ChatPromptTemplate
from qdrant_client import QdrantClient
from qdrant_client.http import models
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich import box
from rich.text import Text

# Constants
LEHMAN_CIK = "806085"
set_identity("your_email")

# API key from Colab secrets, or from the GOOGLE_API_KEY env variable when running locally
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")
except ImportError:
    assert "GOOGLE_API_KEY" in os.environ, "set GOOGLE_API_KEY before running"

console = Console()
print("ok")

ok


In [3]:
def fetch_lehman_filings():
    lehman = Company(LEHMAN_CIK)
    forms = ["10-K", "10-Q"]
    target_filings = []

    start_date = "2008-07-01"

    for form in forms:
        filings = lehman.get_filings(form=form)
        for f in filings:
            # f.filing_date is usually a string like 'YYYY-MM-DD' or a date object
            f_date_str = str(f.filing_date)
            if f_date_str >= start_date:
                target_filings.append(f)

    return target_filings

filings = fetch_lehman_filings()
console.print(f"Found {len(filings)} filings from July 2008 onwards to analyze.")
for f in filings:
    console.print(f"  • {f.form} filed on {f.filing_date}")
# same filing as part 1

Found 1 filings from July 2008 onwards to analyze.

• 10-Q filed on 2008-07-10

In [4]:
def prepare_vector_store(filings):
    if not filings:
        return None

    all_chunks = []
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=8192,
        chunk_overlap=1024
    )

    for f in filings:
        console.print(f"Extracting text from {f.form} ({f.filing_date})...")
        try:
            text = f.text()
            chunks = text_splitter.create_documents(
                texts=[text],
                metadatas=[{"form": f.form, "date": str(f.filing_date), "accession": f.accession_no}]
            )
            all_chunks.extend(chunks)
        except Exception as e:
            console.print(f"[red]Error extracting text from {f.accession_no}: {e}[/red]")

    embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

    # Initialize in-memory Qdrant
    client = QdrantClient(":memory:")
    client.create_collection(
        collection_name="lehman_filings",
        vectors_config=models.VectorParams(size=3072, distance=models.Distance.COSINE),
    )

    vector_store = QdrantVectorStore(
        client=client,
        collection_name="lehman_filings",
        embedding=embeddings,
    )

    # Precise Rate Limiting Logic
    TPM_LIMIT = 30000
    RPM_LIMIT = 100

    tokens_sent_in_minute = 0
    requests_sent_in_minute = 0
    window_start = time.time()

    console.print(f"Indexing {len(all_chunks)} chunks with a strict 30k TPM limit...")

    # separate name so the Qdrant client is not overwritten
    genai_client = genai.Client()

    for i, chunk in enumerate(all_chunks):
        # Precise token count using Gemini SDK
        chunk_tokens = genai_client.models.count_tokens(model="gemini-3-flash-preview", contents=chunk.page_content).total_tokens

        # Check if we would exceed either limit (TPM or RPM)
        if (tokens_sent_in_minute + chunk_tokens > TPM_LIMIT) or (requests_sent_in_minute + 1 > RPM_LIMIT):
            # Calculate precise wait time until the minute resets
            elapsed = time.time() - window_start
            wait_time = max(60 - elapsed, 0) + 1 # +1 for safety margin

            console.print(f"  [yellow]Limit reached ({tokens_sent_in_minute} tokens). Waiting {wait_time:.1f}s for reset...[/yellow]")
            time.sleep(wait_time)

            # Reset window
            window_start = time.time()
            tokens_sent_in_minute = 0
            requests_sent_in_minute = 0

        # Add the document
        vector_store.add_documents([chunk])

        # Update counters
        tokens_sent_in_minute += chunk_tokens
        requests_sent_in_minute += 1

        if (i + 1) % 5 == 0:
            console.print(f"  Indexed {i+1}/{len(all_chunks)} chunks... ({tokens_sent_in_minute} tokens in current window)")

    return vector_store

vector_store = prepare_vector_store(filings)
if vector_store:
    console.print(f"Vector store initialized and indexed.")
# 77 chunks of 8192 chars (~1,500 tokens each), ~4 min mostly waiting for the free-tier limit (30k tokens/min)
# quotas: https://aistudio.google.com/rate-limit
# embeddings rather than keywords: a funding problem can be written "reduced access to secured financing"
# without the word liquidity. for exact names like "Repo 105" keywords would do better
# big chunks keep numbers next to their table headers, but only 10 of them reach the model

Extracting text from 10-Q (2008-07-10)...

Indexing 77 chunks with a strict 30k TPM limit...

Indexed 5/77 chunks... (6427 tokens in current window)

Indexed 10/77 chunks... (14954 tokens in current window)

Indexed 15/77 chunks... (22422 tokens in current window)

Indexed 20/77 chunks... (29495 tokens in current window)

... (12 outputs left out)

Indexed 70/77 chunks... (21169 tokens in current window)

Indexed 75/77 chunks... (29430 tokens in current window)

Limit reached (29430 tokens). Waiting 55.3s for reset...

Vector store initialized and indexed.

In [5]:
class BankruptcyWarningSign(BaseModel):
    warning_type: str = Field(description="Type of warning (e.g., Leverage, Exposure, Capital, Liquidity)")
    key_metric: str = Field(description="Specific financial metric or event")
    value: str = Field(description="Quantified value if available")
    severity: int = Field(description="Severity score 1-10", ge=1, le=10)
    description: str = Field(description="Detailed explanation of the concern")
    exact_passage: str = Field(description="Supporting quote from the filing")

class InvestigationReport(BaseModel):
    findings: List[BankruptcyWarningSign]
    summary: str


# Change the model in case of quota exceeded
# model = "gemini-2.5-flash"
model = "gemini-3-flash-preview"

llm = ChatGoogleGenerativeAI(model=model, temperature=0)
structured_llm = llm.with_structured_output(InvestigationReport)

def run_investigation(vector_store):
    if not vector_store:
        return None, ""

    queries = [
        "Analyze leverage ratios and debt levels. Look for high debt-to-equity or net leverage ratios.",
        "Investigate mortgage-related exposure, specifically subprime and commercial mortgage-backed securities.",
        "Search for mentions of capital raises, liquidity concerns, or funding difficulties.",
        "Identify significant losses in quarterly earnings, especially in the capital markets segment.",
        "Look for Repo 105 or other off-balance sheet accounting treatments used to manage leverage."
    ]

    # take the results rank by rank so each query gets some chunks, and remove
    # duplicates without set() so the order is the same on every run
    results = [vector_store.similarity_search(query, k=5) for query in queries]
    ranked = [doc.page_content for rank in zip(*results) for doc in rank]
    combined_context = "\n---\n".join(list(dict.fromkeys(ranked))[:10])

    prompt = ChatPromptTemplate.from_template("""
    You are a financial forensic analyst. Based ONLY on the following context from Lehman Brothers' 2008 filings,
    identify specific bankruptcy warning signs.

    Context:
    {context}

    Focus on numbers, specific disclosures, and risk factors that preceded the bankruptcy.
    """)

    chain = prompt | structured_llm
    report = chain.invoke({"context": combined_context})
    return report, combined_context

report, rag_context = run_investigation(vector_store)
# exact quotes asked so every finding can be checked (last cell)

In [6]:
def display_report(report):
    if not report:
        console.print("[red]No report generated. check vector store initialization.[/red]")
        return

    console.print(Panel(Text("LEHMAN BROTHERS BANKRUPTCY INVESTIGATION REPORT", style="bold white on red"), box=box.DOUBLE))

    table = Table(title="Detected Warning Signs", box=box.ROUNDED)
    table.add_column("Type", style="cyan")
    table.add_column("Metric", style="magenta")
    table.add_column("Value", style="green")
    table.add_column("Sev", justify="center")
    table.add_column("Description", style="white")

    for finding in report.findings:
        sev_color = "red" if finding.severity >= 8 else "yellow" if finding.severity >= 5 else "green"
        table.add_row(
            finding.warning_type,
            finding.key_metric,
            finding.value,
            f"[{sev_color}]{finding.severity}[/{sev_color}]",
            finding.description
        )

    console.print(table)
    console.print(Panel(report.summary, title="Executive Summary", border_style="blue"))

    console.print("\n[bold]Supporting Passages:[/bold]")
    for i, finding in enumerate(report.findings):
        console.print(f"\n{i+1}. [italic]'{finding.exact_passage}'[/italic]")

display_report(report)
# 7 signs: $2.77bn loss, $17.9bn cash used in operations, -$3.0bn fixed income revenues,
# $2.9bn collateral if downgraded, leverage 24.3, $23.9bn high yield, $60.8bn mortgage / asset-backed

╔═════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
║ LEHMAN BROTHERS BANKRUPTCY INVESTIGATION REPORT                                                                 ║
╚═════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝

                                              Detected Warning Signs                                               
╭───────────┬─────────────────────────────────────┬──────────────────┬─────┬──────────────────────────────────────╮
│ Type      │ Metric                              │ Value            │ Sev │ Description                          │
├───────────┼─────────────────────────────────────┼──────────────────┼─────┼──────────────────────────────────────┤
│ Capital   │ Net Income                          │ $(2,774) million │ 10  │ The company reported a massive net   │
│           │                                     │                  │     │ loss for the quarter ended May 31,   │
│           │                                     │                  │     │ 2008, representing a catastrophic    │
│           │                                     │                  │     │ swing from the $4.19 billion profit  │
│           │                                     │                  │     │ reported in the prior fiscal year    │
│           │                                     │                  │     │ end.                                 │
│ Liquidity │ Net cash used in operating          │ $(17.9) billion  │ 10  │ A severe drain on operational        │
│           │ activities                          │                  │     │ liquidity is evident, with nearly    │
│           │                                     │                  │     │ $18 billion in cash consumed by      │
│           │                                     │                  │     │ operations in just six months,       │
│           │                                     │                  │     │ forcing the firm to rely heavily on  │
│           │                                     │                  │     │ financing activities to stay afloat. │
│ Exposure  │ Fixed Income Net Revenues           │ $(2,975) million │  9  │ The Fixed Income segment, previously │
│           │                                     │                  │     │ a profit center, suffered a total    │
│           │                                     │                  │     │ collapse in revenue due to negative  │
│           │                                     │                  │     │ valuation adjustments on mortgage    │
│           │                                     │                  │     │ and asset-backed positions.          │
│ Liquidity │ Collateral Posting Requirements     │ $2.9 billion     │  8  │ Credit rating downgrades by S&P and  │
│           │                                     │                  │     │ Fitch triggered immediate            │
│           │                                     │                  │     │ contractual requirements to post     │
│           │                                     │                  │     │ billions in additional collateral,   │
│           │                                     │                  │     │ creating a liquidity death spiral.   │
│ Leverage  │ Leverage Ratio                      │ 24.34x           │  8  │ Despite attempts to deleverage, the  │
│           │                                     │                  │     │ firm maintained a high leverage      │
│           │                                     │                  │     │ ratio, meaning even small declines   │
│           │                                     │                  │     │ in asset value could (and did) wipe  │
│           │                                     │                  │     │ out the equity base.                 │
│ Exposure  │ High Yield Instruments              │ $23.856 billion  │  7  │ The firm held significant positions  │
│           │                                     │                  │     │ in non-investment grade 'high yield' │
│           │                                     │                  │     │ debt, which are highly sensitive to  │
│           │                                     │     

╭─────────────────────────────────────────────── Executive Summary ───────────────────────────────────────────────╮
│ Lehman Brothers' 2008 filings reveal a firm in a state of terminal decline. The most critical warning signs     │
│ include a $2.77 billion quarterly net loss, a staggering $17.9 billion negative cash flow from operations, and  │
│ a collapse in Fixed Income revenues driven by toxic mortgage-backed assets. The firm's high leverage and        │
│ reliance on short-term secured funding (repo) markets made it vulnerable to the credit rating downgrades that   │
│ occurred in June 2008, which triggered massive collateral calls and exhausted the firm's liquidity pool shortly │
│ before its bankruptcy filing.                                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Supporting Passages:

1. 'Net income (2,774  Three Months Ended May 31, 2008'

2. 'net cash used in operating activities of $17.9 billion and net cash used in investing activities of $0.6 
billion were partially offset by net cash provided by financing activities of $17.7 billion.'

3. 'Net revenues in Capital Markets - Fixed Income were negative $3.0 billion and negative $2.7 billion for the 
2008 three and six months, respectively'

4. 'counterparties to derivative contracts and other secured funding arrangements would have the right to require 
the Company to post additional collateral... of approximately $2.9 billion in the event the Company was to 
experience a further downgrade'

5. 'Leverage ratio 24.34  May 31, 2008'

6. 'High yield instruments 23,856  May 31, 2008'

7. 'Mortgage and asset- backed securities (2) $ 60.8  May 31, 2008'

In [7]:
import re

def normalize(s):
    return re.sub(r"\s+", " ", s).strip().lower()

def check_quotes(report, source_text):
    # look for each quote in the text given to the model
    src = normalize(source_text)
    ok = 0
    for i, f in enumerate(report.findings, 1):
        found = normalize(f.exact_passage) in src
        ok += found
        console.print(f"{i}. {'found' if found else 'NOT FOUND'} - {f.key_metric}")
    console.print(f"{ok}/{len(report.findings)} quotes found in the source text")
    return ok, len(report.findings)

check_quotes(report, rag_context)
# only 2/7 word for word. 4 are table rows rewritten as a line ("Leverage ratio 24.34 May 31, 2008"),
# 1 is a real sentence cut with "...". checked in the PDF: all the numbers are real, the quotes are not
# vs part 1: ~15k tokens instead of ~110k, but 7 signs instead of 10 and 2/7 quotes instead of 7/10.
# missed the june share issue and the level 3 assets, probably not in the 10 chunks
# todo: accept "..." in the check (test each piece), keep tables intact when chunking

1. NOT FOUND - Net Income

2. found - Net cash used in operating activities

3. found - Fixed Income Net Revenues

4. NOT FOUND - Collateral Posting Requirements

5. NOT FOUND - Leverage Ratio

6. NOT FOUND - High Yield Instruments

7. NOT FOUND - Mortgage and asset-backed securities

2/7 quotes found in the source text

(2, 7)